In [15]:
from dotenv import load_dotenv
from io import StringIO
load_dotenv()
import os
import pandas as pd
import json
from typing import Dict, Any, List
from langchain_google_genai import ChatGoogleGenerativeAI  # UPDATED: Google Gemini LLM
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate
from langchain.tools import tool
from langchain_core.runnables import RunnableSequence
from langchain_community.document_loaders import (
    TextLoader, PyPDFLoader
)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_neo4j import Neo4jGraph
from neo4j import GraphDatabase

# Env & Graph Setup (UPDATED: Google API Key)
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")  # NEW: Google key instead of Groq

if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, GOOGLE_API_KEY]):
    raise ValueError("Missing env vars (add GOOGLE_API_KEY to .env).")

class CustomNeo4jGraph(Neo4jGraph):
    def query(self, query, params=None, **kwargs):
        with self._driver.session(**kwargs) as session:
            result = session.run(query, params or {})
            return [dict(record) for record in result]

graph = CustomNeo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="neo4j",
    refresh_schema=False
)

# UPDATED: LLM to Google Gemini (replaces Groq)
llm = ChatGoogleGenerativeAI(
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-pro",  # FIXED: Valid Gemini model (strong for agents/extraction)
    temperature=0
)

def clear_all(tx): 
    tx.run("MATCH (n) DETACH DELETE n")

direct_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# FIXED Tool 1: load_file (robust unescaping)
@tool
def load_file(file_path: str) -> str:
    """Load any file (CSV, JSON, TXT, PDF) and return processed content as JSON."""
    try:
        # FIXED: Robust unescape for malformed inputs (e.g., "new_article.txt\"" → new_article.txt)
        if isinstance(file_path, str):
            file_path = file_path.strip().strip('"').replace('\\"', '"').replace('\\\\', '\\')
        file_ext = os.path.splitext(file_path)[1].lower()
        if file_ext == '.csv':
            df = pd.read_csv(file_path)
            content = df.to_json(orient='records')
            return json.dumps({'type': 'structured', 'content': content, 'metadata': f"CSV: {len(df)} rows, {len(df.columns)} cols: {list(df.columns)}"})
        elif file_ext == '.json':
            with open(file_path, 'r') as f:
                data = json.load(f)
            df = pd.json_normalize(data) if isinstance(data, list) else pd.DataFrame([data])
            content = df.to_json(orient='records')
            return json.dumps({'type': 'structured', 'content': content, 'metadata': f"JSON: {len(df)} rows, cols: {list(df.columns)}"})
        elif file_ext in ['.txt', '.pdf']:
            if file_ext == '.pdf':
                loader = PyPDFLoader(file_path)
            else:
                loader = TextLoader(file_path, encoding='utf-8')
            docs = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
            chunks = text_splitter.split_documents(docs)
            content = [chunk.page_content for chunk in chunks]
            return json.dumps({'type': 'unstructured', 'content': content, 'metadata': f"{file_ext.upper()}: {len(chunks)} chunks, total chars: {sum(len(c) for c in content)}"})
        else:
            return json.dumps({'error': f"Unsupported: {file_ext}. Use CSV/JSON/TXT/PDF."})
    except Exception as e:
        return json.dumps({'error': f"Load failed: {str(e)[:200]}. Path: '{file_path}' (check for extra quotes)."})

# FIXED Tool 2: analyze_data (StringIO, str conversion)
@tool
def analyze_data(loaded_json: str) -> str:
    """Analyze loaded file JSON: For structured, columns/types/sample; For unstructured, chunks overview."""
    try:
        if isinstance(loaded_json, str):
            data_str = loaded_json.strip().strip('"').replace('\\"', '"')
            data = json.loads(data_str)
        else:
            data = loaded_json
        if 'error' in data:
            return json.dumps(data)
        if data['type'] == 'structured':
            df = pd.read_json(StringIO(data['content']))
            for col in df.columns:
                if pd.api.types.is_datetime64_any_dtype(df[col]):
                    df[col] = df[col].astype(str)
                elif df[col].dtype == 'object':
                    df[col] = df[col].astype(str).str.strip()
            numeric_cols = df.select_dtypes(include=['object']).columns
            for col in numeric_cols:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            df = df.dropna(subset=[col for col in df.columns if df[col].nunique() > 1])
            head_sample = df.head(2).applymap(str).to_dict('records')
            columns_types = {col: str(df[col].dtype) for col in df.columns}
            entities = [col for col in df.columns if any(kw in col.lower() for kw in ['id', 'name', 'person', 'company', 'product', 'date'])]
            summary = {
                "shape": list(df.shape),
                "columns_types": columns_types,
                "head_sample": head_sample,
                "potential_entities": entities,
                "potential_rels": "Look for shared keys (e.g., 'id' linking 'customer' and 'order')."
            }
            return json.dumps(summary, ensure_ascii=False, indent=2)
        else:
            chunks = data['content'][:5]
            summary = {
                "num_chunks": len(data['content']),
                "sample_chunks": [c[:150] + "..." for c in chunks],
                "word_count": sum(len(c.split()) for c in data['content'])
            }
            return json.dumps(summary, ensure_ascii=False, indent=2)
    except json.JSONDecodeError as e:
        return json.dumps({'error': f"JSON parse failed: {str(e)[:100]}. Input may be unescaped—retry with clean JSON."})
    except Exception as e:
        return json.dumps({'error': f"Analysis failed: {str(e)[:100]}. Check data format."})

# Extraction Prompt (unchanged, no braces issues)
extraction_prompt = PromptTemplate(
    input_variables=["chunk"],
    template="""Extract entities and relationships from text as triples. Entities: Person (names), Organization (companies), Product/Item, Event/Date.
Relationships: WORKS_AT, BUYS, LOCATED_IN, MENTIONS, etc. Include props (e.g., role:str, amount:float).

Few-shot 1: "Alice Johnson is a software engineer at Neo4j in San Francisco since 2023." -> [{'subject': 'Alice Johnson', 'rel': 'WORKS_AT', 'obj': 'Neo4j', 'props': {'role': 'software engineer', 'location': 'San Francisco', 'since': '2023'}}, {'subject': 'Neo4j', 'rel': 'LOCATED_IN', 'obj': 'San Francisco', 'props': {}}]

Few-shot 2: "Bob bought a laptop for $1200 on 2025-01-15." -> [{'subject': 'Bob', 'rel': 'BUYS', 'obj': 'laptop', 'props': {'amount': 1200.0, 'date': '2025-01-15'}}]

Few-shot 3: "The event on Oct 19, 2025, features AI topics." -> [{'subject': 'event', 'rel': 'HAS_DATE', 'obj': 'Oct 19, 2025', 'props': {}}, {'subject': 'event', 'rel': 'MENTIONS', 'obj': 'AI', 'props': {'topic': True}}]

Chunk: {chunk}
Output ONLY JSON list of dicts (3-5 triples max, no dups). Cast numbers as float."""
)

# FIXED: Extraction Chain (RunnableSequence)
extraction_chain = extraction_prompt | llm

@tool
def extract_entities_and_rels(chunks_json: str) -> str:
    """Extract and dedup triples from unstructured chunks JSON."""
    try:
        if isinstance(chunks_json, str):
            data_str = chunks_json.strip().strip('"').replace('\\"', '"')
            data = json.loads(data_str)
        else:
            data = chunks_json
        if data['type'] != 'unstructured':
            return json.dumps({'error': "For unstructured files only."})
        chunks = data['content'][:5]
        all_triples = []
        seen = set()
        for chunk in chunks:
            # FIXED: .invoke and .text
            triples_response = extraction_chain.invoke({"chunk": chunk})
            triples_str = triples_response.content  # UPDATED: Gemini uses .content (not .text)
            triples = json.loads(triples_str)
            for t in triples:
                key = (t['subject'], t['rel'], t['obj'])
                if key not in seen:
                    seen.add(key)
                    all_triples.append(t)
        for t in all_triples:
            t['count'] = sum(1 for tt in all_triples if tt['subject'] == t['subject'] and tt['rel'] == t['rel'] and tt['obj'] == t['obj'])
        return json.dumps(all_triples, ensure_ascii=False, indent=2)
    except json.JSONDecodeError as e:
        return json.dumps({'error': f"JSON parse failed in extraction: {str(e)[:100]}"})
    except Exception as e:
        return json.dumps({'error': f"Extraction failed: {str(e)[:100]}"})

# FIXED: Enhanced schema_prompt (better classification, escaped if needed)
schema_prompt = PromptTemplate(
    input_variables=["input_data"],
    template="""Infer Neo4j schema from structured analysis JSON or unstructured triples. Classify labels semantically.

For structured: Nodes from columns (e.g., 'name'/'id' -> Person with unique name; 'product' -> Product).

For unstructured triples: 
- Classify subjects/objs: Person (human names like 'Dr. Elena', 'Alice'); Organization (companies/universities like 'Neo4j', 'University of GreenTech'); Article/Study (research terms); Product (items); Event (dates/announcements).
- Nodes: Unique labels with 'name: str' unique prop; Add type-specific props (e.g., Organization: {{'location': 'str'}}).
- Rels: From 'rel' keys; Props from 'props' dict (infer types).

Few-shot Unstructured: [{{'subject': 'Dr. Elena Martinez', 'rel': 'AFFILIATEDWITH', 'obj': 'University of GreenTech', 'props': {{'role': 'researcher'}}, 'count': 1}}, {{'subject': 'Study', 'rel': 'DISCOVERED', 'obj': 'Policy changes', 'props': {{'field': 'AI'}}, 'count': 1}}] 
-> {{"nodes": [{{'label': "Person", 'properties': [{{'name': "name", 'type': "str", 'unique': true}}]}}, {{'label': "Organization", 'properties': [{{'name': "name", 'type': "str", 'unique': true}}]}}, {{'label': "Study", 'properties': [{{'name': "name", "type": "str"}}]}}, {{'label': "Policy", 'properties': [{{'name': "name", "type": "str"}}]}}, "relationships": [{{'type': "AFFILIATEDWITH", 'from': "Person", 'to': "Organization", 'props': [{{'name': "role", "type": "str"}}]}}, {{'type': "DISCOVERED", 'from': "Study", 'to': "Policy", 'props': [{{'name': "field", "type": "str"}}]}}]}}

Few-shot Structured: {{"potential_entities": ["customer_name", "product_id"]}} -> {{"nodes": [{{'label': "Customer", 'properties': [{{'name': "name", "type": "str", 'unique': true}}, {{'name': "id", "type": "int", 'unique': true}}]}}, {{'label': "Product", 'properties': [{{'name': "id", "type": "int", 'unique': true}}, {{'name': "name", "type": "str"}}]}}, "relationships": [{{'type': "PURCHASED", 'from': "Customer", 'to': "Product"}}]}}

Input: {input_data}
Output ONLY valid JSON schema (nodes/relationships as lists; classify labels accurately, avoid generic 'Entity')."""
)

# FIXED: Schema Chain (RunnableSequence)
schema_chain = schema_prompt | llm

@tool
def infer_schema(input_data: str) -> str:
    """Infer schema from analysis JSON (structured) or triples JSON (unstructured)."""
    try:
        if isinstance(input_data, str):
            data_str = input_data.strip().strip('"').replace('\\"', '"')
            schema_response = schema_chain.invoke({"input_data": data_str})
            schema_json = schema_response.content  # UPDATED: Gemini uses .content
        else:
            schema_response = schema_chain.invoke({"input_data": input_data})
            schema_json = schema_response.content
        return schema_json
    except json.JSONDecodeError as e:
        return json.dumps({'error': f"JSON parse failed in schema: {str(e)[:100]}"})
    except Exception as e:
        return json.dumps({'error': f"Schema inference failed: {str(e)[:100]}"})

@tool
def generate_import_cypher(schema_str: str, data_str: str) -> str:
    """Generate Cypher: Structured (LOAD CSV MERGE); Unstructured (UNWIND triples MERGE with labels/props)."""
    try:
        schema_clean = schema_str.strip().strip('"').replace('\\"', '"') if isinstance(schema_str, str) else schema_str
        data_clean = data_str.strip().strip('"').replace('\\"', '"') if isinstance(data_str, str) else data_str
        schema = json.loads(schema_clean)
        data = json.loads(data_clean)
        if 'error' in data or 'error' in schema:
            return json.dumps({'error': "Invalid data/schema."})
        cypher_parts = []

        # Constraints
        for node in schema['nodes']:
            for prop in node['properties']:
                if prop.get('unique'):
                    cypher_parts.append(f"CREATE CONSTRAINT {node['label']}_{prop['name']} IF NOT EXISTS FOR (n:{node['label']}) REQUIRE n.{prop['name']} IS UNIQUE;")

        if 'type' in data and data['type'] == 'structured':
            df = pd.read_json(StringIO(data['content']))
            df = df.astype({col: str for col in df.select_dtypes(include=['datetime64'])})
            df.to_csv('/tmp/temp.csv', index=False)
            load_csv = "LOAD CSV WITH HEADERS FROM 'file:///temp.csv' AS row\n"
            for node in schema['nodes']:
                label = node['label']
                unique_prop = next((p['name'] for p in node['properties'] if p.get('unique')), None)
                match = f"(n:{label} {{ {unique_prop}: toInteger(row.{unique_prop}) }})" if unique_prop else f"(n:{label})"
                props_str = ', '.join([
                    f"n.{p['name']} = CASE WHEN row.{p['name']} =~ '^\\\\d+(\\\\.\\\\d+)?$' THEN toFloat(row.{p['name']}) ELSE row.{p['name']} END"
                    for p in node['properties'] if p['name'] in df.columns
                ])
                cypher_parts.append(f"{load_csv}MERGE {match} SET {props_str};")
            for rel in schema['relationships']:
                link_prop = "id"
                cypher_parts.append(f"{load_csv}MATCH (from:{rel['from']}), (to:{rel['to']}) WHERE toString(from.{link_prop}) = toString(row.{link_prop}) MERGE (from)-[:{rel['type']}]->(to);")
        else:
            triples = data['content']
            cypher_parts.append("UNWIND $triples AS t")
            for triple in triples:
                subj_label = next((n['label'] for n in schema['nodes'] if 'name' in [p['name'] for p in n['properties']]), "Entity")
                obj_label = subj_label
                rel = triple['rel']
                subj_match = f"(s:{subj_label} {{name: t.subject}})"
                obj_match = f"(o:{obj_label} {{name: t.obj}})"
                props_str = ', '.join([f"r.{k} = t.props.{k}" for k in triple['props']])
                cypher_parts.append(f"MERGE {subj_match} SET s.name = t.subject; MERGE {obj_match} SET o.name = t.obj; MERGE (s)-[r:{rel}]->(o) SET {props_str};")
            cypher_parts[-1] += f"\n // Params: {{'triples': {json.dumps(triples, ensure_ascii=False)}}}"

        full_cypher = "\n".join(cypher_parts)
        return full_cypher
    except Exception as e:
        return json.dumps({'error': f"Cypher gen failed: {str(e)[:200]}"})

@tool
def execute_cypher(cypher: str, data_json: str = None) -> str:
    """Execute Cypher (params from data_json if unstructured). Return detailed validation."""
    try:
        with direct_driver.session() as session:
            session.execute_write(clear_all)
        params = {}
        if data_json:
            params_str = data_json.strip().strip('"').replace('\\"', '"')
            params = json.loads(params_str) if params_str else {}
        result = graph.query(cypher, params=params)
        node_query = "MATCH (n) RETURN labels(n)[0] as label, count(n) as count GROUP BY label"
        rel_query = "MATCH ()-[r]->() RETURN type(r) as type, count(r) as count GROUP BY type"
        nodes = graph.query(node_query)
        rels = graph.query(rel_query)
        node_summary = [{'label': r['label'], 'count': r['count']} for r in nodes]
        rel_summary = [{'type': r['type'], 'count': r['count']} for r in rels]
        return f"Success: Executed. Nodes: {node_summary}. Rels: {rel_summary}. Total nodes: {sum(r['count'] for r in nodes) if nodes else 0}, Rels: {sum(r['count'] for r in rels) if rels else 0}."
    except Exception as e:
        return f"Execution failed: {str(e)[:200]}. Check Cypher syntax or params."

# FIXED Agent Prompt (rewritten with no unescaped braces; standard ReAct)
agent_prompt = PromptTemplate(
    input_variables=["input", "agent_scratchpad"],
    template="""You are an Enhanced Multi-File Graph Builder Agent. Adapt to file type from load_file output.

You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of {tool_names}
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Observation repeats until final answer)
Thought: [Final reasoning]
Action: Final Answer
Action Input: [Your response here]

Tools flow guidance:
- Start with load_file(file_path) where file_path is plain string like 'new_article.txt'.
- If structured (CSV/JSON): analyze_data -> infer_schema -> generate_import_cypher -> execute_cypher.
- If unstructured (TXT/PDF): analyze_data -> extract_entities_and_rels -> infer_schema -> generate_import_cypher -> execute_cypher.
- Clean inputs: No extra quotes/braces in Action Input for paths/JSON. If error (e.g., path), retry unquoted.
- Goal: Build/validate graph. Stop on success or 3 errors.

Question: {input}
{agent_scratchpad}"""
)

tools = [load_file, analyze_data, extract_entities_and_rels, infer_schema, generate_import_cypher, execute_cypher]
agent = create_react_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=25, handle_parsing_errors=True, return_intermediate_steps=True)

def create_multi_file_graph_agent(file_path: str) -> AgentExecutor:
    return agent_executor

# Test
if __name__ == "__main__":
    test_file = "news_article.txt"  # Or "sales_sample.csv"
    agent_exec = create_multi_file_graph_agent(test_file)
    task = f"Build Neo4j graph from {test_file}. Handle type, infer schema, execute."
    result = agent_exec.invoke({"input": task})
    print("Final Result:", result['output'])




> Entering new AgentExecutor chain...


ChatGoogleGenerativeAIError: Invalid argument provided to Gemini: 400 API Key not found. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API Key not found. Please pass a valid API key."
]